<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [14]:
import pandas as pd
import matplotlib.pyplot as plt

# Download the dataset from the GitHub repository
url = "https://raw.githubusercontent.com/shivashankarb2006/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Data shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Data shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [15]:
# Create an observed decline flag
df["is_declining"] = (
    df["impressions_last_30d"] <
    0.8 * df["impressions_prev_30d"]
).astype(int)

print("Declining pages:", df["is_declining"].sum())
print("Non-declining pages:", (df["is_declining"] == 0).sum())

# Signal 1: previous-period impressions
print("\nSignal 1: Previous-period impressions")
print(
    df.groupby("is_declining")["impressions_prev_30d"]
    .median()
    .round(2)
)

# Signal 2: average position
print("\nSignal 2: Average position")
print(
    df.groupby("is_declining")["avg_position"]
    .median()
    .round(2)
)

# Signal 3: content freshness
print("\nSignal 3: Days since last update")
print(
    df.groupby("is_declining")["days_since_last_update"]
    .median()
    .round(2)
)

Declining pages: 16262
Non-declining pages: 13738

Signal 1: Previous-period impressions
is_declining
0    103.5
1    313.0
Name: impressions_prev_30d, dtype: float64

Signal 2: Average position
is_declining
0    10.05
1    11.30
Name: avg_position, dtype: float64

Signal 3: Days since last update
is_declining
0    20.0
1    20.0
Name: days_since_last_update, dtype: float64


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [16]:
# Test whether trend direction agrees with observed decline

flag_test = pd.crosstab(
    df["trend_direction"],
    df["is_declining"],
    normalize="index"
).round(3)

print("Trend direction vs observed decline:")
print(flag_test)

Trend direction vs observed decline:
is_declining       0    1
trend_direction          
down             0.0  1.0
flat             1.0  0.0
new              1.0  0.0
stable           1.0  0.0
up               1.0  0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal tests show which historical content signals are associated with observed impression decline. These patterns should be treated as directional evidence for prioritization rather than proof that a specific content change caused the decline.

A content team can use these signals to identify pages for human review, while avoiding automatic optimization decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.